<a href="https://colab.research.google.com/github/Aswanth0704/gpu-programming-cpp/blob/main/2_Extended_Algorithms.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [33]:
import os

if os.getenv("COLAB_RELEASE_TAG"): # If running in Google Colab:
  !mkdir -p Sources
  !wget https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.03-Extending-Algorithms/Sources/ach.h -nv -O Sources/ach.h

2026-09-06 03:22:18 URL:https://raw.githubusercontent.com/NVIDIA/accelerated-computing-hub/refs/heads/main/tutorials/cuda-cpp/notebooks/01.03-Extending-Algorithms/Sources/ach.h [2787/2787] -> "Sources/ach.h" [1]


Find maximum temperature in each step.

In [2]:
%%writefile Sources/naive-max-diff.cpp

#include "ach.h"
float naive_max_change(
  const thrust::universal_vector<float>& a,
  const thrust::universal_vector<float>& b
)
{
  // allocate a vector to store 'a'-'b'
  thrust::universal_vector<float> unnecessarily_materialized_diff(a.size());

  // compute difference:
  thrust::transform(thrust::device,
                    a.begin(), a.end(), // first input sequence
                    b.begin(),          // second input sequence
                    unnecessarily_materialized_diff.begin(), // store here
                    []__host__ __device__ (float x, float y){
                      return abs(x - y); //
                    });

  // compute max difference
  return thrust::reduce(thrust::device,
                        unnecessarily_materialized_diff.begin(),
                        unnecessarily_materialized_diff.end(),
                        0.0f,
                        thrust::maximum<float>{});

}

int main()
{
  float k = 0.5;
  float ambient_temp = 20;
  thrust::universal_vector<float> temp[] = {{42, 24, 50}, {0, 0, 0}};
  auto transformation = [=]__host__ __device__ (float temp){
    return temp + k*(ambient_temp - temp);
  };

  std::printf("step   max-change\n");
  for(int step = 0; step < 3; step ++){
    thrust::universal_vector<float> &current = temp[step % 2];
    thrust::universal_vector<float> &next = temp[(step +1)%2];

    thrust::transform(thrust::device, current.begin(), current.end(), next.begin(), transformation);
    std::printf("%d   %.2f\n", step, naive_max_change(current, next));
  }
}

Writing Sources/naive-max-diff.cpp


In [3]:
!nvcc --extended-lambda -o /tmp/a.out Sources/naive-max-diff.cpp -x cu -arch=native
!/tmp/a.out

step   max-change
0   15.00
1   7.50
2   3.75


**Problem with the above code**
- We started allocating storage for differences.
- We read 2*n floats from both a and b, write n elements back to memory.
- As part of the reduction step, we load n integers.
- Therefore, we have 4*n memory accesses.
- We could implement this in about 2*n memory access with a single for loop.

In [ ]:

# float max_diff = 0.;
# for(int i = 0; i < a.size(); i++){
#     max_diff = std::max(max_diff, std::abs(a[i] - b[i]))
# }


- Now we have a two-fold reduction in amount of memory accesses should result in about two fold speedup. 2x speedup and save space on GPU by avoiding to save the array.

**Iterators**
- A pointer, int* pointer, points to a sequence of integers in memory.
- We can dereference a pointer to get access to the integer it currently points to.
- We can advance pointer with pointer++ to make it point to the next element in the sequence.


In [4]:
%%writefile Sources/pointer.cpp
#include "ach.h"
int main(){
  std::array<int, 3> a{0, 3, 5};
  int *pointer = a.data();
  std::printf("pointer[0]: %d\n", pointer[0]); // 0
  std::printf("pointer[1]: %d\n", pointer[1]); // 3
}

Writing Sources/pointer.cpp


In [ ]:
!nvcc --extended-lambda -o /tmp/a.out Sources/pointer.cpp -x cu -arch=native
!/tmp/a.out

pointer[0]: 0
pointer[1]: 3


# Operator overloading:
- In C++ we can define what operators such as *, ++ ,[] do. Concept of iterators build on the top of this idea.



### Simple counter
- We want to create an infinite sequence without allocating a single byte of memory. We will redefine the square brackets [] operator.

Let's first start with iterators:
- Iterators are used to access and iterate through elements of datastructure like vectors, maps, sets by pointing to them.

In [5]:
%%writefile Sources/iterators.cpp
#include<iostream>
#include<vector>

using namespace std;

int main(){
  vector<string> cars = {"Volvo", "BMW", "Ford", "Mazda"};
  // create an iterator called it:
  vector<string>::iterator it;

  // use the iterator to loop through the vector
  // begin() returuns an iterator that points to the first element of the data structure.
  // end() returns an iterator that points to one position after the last element.
  for (it = cars.begin(); it != cars.end(); ++it){
    cout<<*it<<"\n"; // *it is dereferencing it.
  }

  // modify the value:
  it = cars.begin();
  *it = "Tesla";
  cout<<"first element now is: "<< cars[0]<< "\n";

  // The auto keyword: Introduced from C++11
  // instead of vector<string>::iterator = cars.begin(),
  // I can use auto = cars.begin()

  // When you just want to read and not modify elements, use for-each loop:
  for(string car: cars){
    cout<<car<<"\n";
  }

  // when you need to modify, add, remove, or skip elements, use iterators.
  for(auto it = cars.begin(); it!= cars.end();){
    if(*it == "BMW"){
      it = cars.erase(it); // erase?
    } else {
      ++it;
    }
  }

  // print
  for(const string &car : cars){ // why use const with memory address
    cout<<car<<"\n";
  }

  // iterate in reverse:
  for (auto it = cars.rbegin(); it !=cars.rend(); ++it){
    cout<<*it<<"\n";
  }

  return 0;
}


Writing Sources/iterators.cpp


In [ ]:
!g++ Sources/iterators.cpp -o /tmp/a.out
!/tmp/a.out

Volvo
BMW
Ford
Mazda
first element now is: Tesla
Tesla
BMW
Ford
Mazda
Tesla
Ford
Mazda
Mazda
Ford
Tesla


Let's get back to the counting iterator
- we want to create infinite sequence without using a single byte.
- Trick is to overload [] operator.

In [6]:
%%writefile Sources/counting.cpp
#include "ach.h"

struct counting_iterator
{
  int operator[](int i){
    return i;
  }
};

int main(){
  counting_iterator it;

  std::printf("it[0]: %d\n", it[0]);
  std::printf("it[1]: %d\n", it[1]);
}

Writing Sources/counting.cpp


In [ ]:
!nvcc --extended-lambda -o /tmp/a.out Sources/counting.cpp -x cu -arch=native
!/tmp/a.out

it[0]: 0
it[1]: 1


## Simple Transform Iterator
- instead of simple counting, we multiple each input value times 2.

In [7]:
%%writefile Sources/transform.cpp
#include "ach.h"

struct transform_iterator
{
  int *a;
  int operator[](int i){
    return a[i]*2;
  }
};

int main(){
  std::array<int, 3> a{0, 1, 2};
  transform_iterator it{a.data()};

  std::printf("it[0]: %d\n", it[0]);
  std::printf("it[1]: %d\n", it[1]);
}

Writing Sources/transform.cpp


In [ ]:
!nvcc --extended-lambda -o /tmp/a.out Sources/transform.cpp -x cu -arch=native
!/tmp/a.out

it[0]: 0
it[1]: 2


## Simple Zip iterator:
- we can redefine the [] operator to combine two sequences

In [8]:
%%writefile Sources/zip.cpp
#include "ach.h"

struct zip_iterator
{
  int *a;
  int *b;
  std::tuple<int, int> operator[](int i){
    return {a[i], b[i]};
  }
};

int main(){
  std::array<int, 3> a{0, 1, 3};
  std::array<int, 3> b{5, 4, 2};

  zip_iterator it{a.data(), b.data()};
  std::printf("it[0]: (%d, %d)\n", std::get<0>(it[0]), std::get<1>(it[0]));
  std::printf("it[1]: (%d, %d)\n", std::get<0>(it[1]), std::get<1>(it[1]));
}

Writing Sources/zip.cpp


In [ ]:
!nvcc --extended-lambda -o /tmp/a.out Sources/zip.cpp -x cu -arch=native
!/tmp/a.out

it[0]: (0, 5)
it[1]: (1, 4)


## Combining input iterators:


In [9]:
%%writefile Sources/transform-zip.cpp
#include "ach.h"

struct zip_iterator
{
  int *a;
  int *b;
  std::tuple<int, int> operator[](int i){
    return {a[i], b[i]};
  }
};

struct transform_iterator
{
  zip_iterator zip;
  int operator[](int i){
    auto [a, b] = zip[i];
    return abs(a-b);
  }
};

int main(){
  std::array<int, 3> a{0, 1, 3};
  std::array<int, 3> b{5, 4, 2};

  zip_iterator zip{a.data(), b.data()};
  transform_iterator it{zip};
  std::printf("it[0]: %d\n", it[0]);
  std::printf("it[1]: %d\n", it[1]);
}



Writing Sources/transform-zip.cpp


In [ ]:
!nvcc --extended-lambda -o /tmp/a.out Sources/transform-zip.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

it[0]: 5
it[1]: 3


## Transform Output Iterator
The concept of iterators is not limited to inputs alone. One can transform values that are written into a transform output iterator. Note: in the code below, both = and [] operators are being redefined.

In [10]:
%%writefile Sources/transform-output.cpp
#include "ach.h"

struct wrapper{
  int *ptr;
  void operator=(int value)
  {
    *ptr = value/2; // assigning the value
  }
};

struct tranform_output_iterator
{
  int *a;
  wrapper operator[](int i)
  {
    return {a + i}; // I didnot understand this {} notation. What does this mean. moving the pointer to the index.
  }
};

int main(){

  std::array<int, 3> a{0, 1, 2};
  tranform_output_iterator it{a.data()};
  it[0] = 10;
  it[1] = 20;

  std::printf("a[0]: %d\n", a[0]);
  std::printf("a[1]: %d\n", a[1]);
}

Writing Sources/transform-output.cpp


In [11]:
!nvcc --extended-lambda -o /tmp/a.out Sources/transform-output.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

a[0]: 5
a[1]: 10


Discard iterator:

In [12]:
%%writefile Sources/discard.cpp
#include "ach.h"

struct wrapper{
  void operator=(int value){
    // discard value
  }
};

struct discard_iterator{
  wrapper operator[](int i){
    return {};
  }
};

int main()
{
  discard_iterator it;
  it[0] = 10;
  it[1] = 20;
}


Writing Sources/discard.cpp


In [13]:
!nvcc --extended-lambda -o /tmp/a.out Sources/discard.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

CUDA fancy iterators:
- thrus::zip_iterator:

In [19]:
%%writefile Sources/zip.cpp

#include "ach.h"

int main()
{
  // allocate ad intialize input vectors:
  thrust::universal_vector<float> a{31, 22, 35};
  thrust::universal_vector<float> b{25, 21, 27};

  // zip two vectors into a single iterator:
  auto zip = thrust::make_zip_iterator(a.begin(), b.begin());

  thrust::tuple<float, float> first = *zip;
  std::printf("first: (%g, %g)\n", thrust::get<0>(first), thrust::get<1>(first));

  zip++;
  thrust::tuple<float, float> second = *zip;
  std::printf("second: (%g, %g)\n", thrust::get<0>(second), thrust::get<1>(second));

}

Overwriting Sources/zip.cpp


In [20]:
!nvcc --extended-lambda -o /tmp/a.out Sources/zip.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

first: (31, 25)
second: (22, 21)


lets use thrust::transform_iterator to find the difference between the two;

In [21]:
%%writefile Sources/tranform.cpp
#include "ach.h"

int main()
{
  thrust::universal_vector<float> a{31, 22, 35};
  thrust::universal_vector<float> b{25, 21, 27};

  auto zip = thrust::make_zip_iterator(a.begin(), b.begin());
  auto tranform = thrust::make_transform_iterator(zip, [] __host__ __device__ (thrust::tuple<float, float> t){
    return abs(thrust::get<0>(t) - thrust::get<1>(t));
  });

  std::printf("first: %g\n", *tranform);
  tranform++;
  std::printf("second: %g\n", *tranform);
}

Writing Sources/tranform.cpp


In [22]:
!nvcc --extended-lambda -o /tmp/a.out Sources/transform.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

it[0]: 0
it[1]: 2


Now let's compute the maximum changes in the temperature at each step using the thrust library and Fancy iterators

In [31]:
%%writefile Sources/optimized-max-diff.cpp

#include "ach.h"

float max_change(const thrust::universal_vector<float> a,
                 const thrust::universal_vector<float> b)
{
  // zip
  auto zip = thrust::make_zip_iterator(a.begin(), b.begin());

  // transform: diff
  auto transform = thrust::make_transform_iterator(zip, []__host__ __device__ (thrust::tuple<float, float> t){
    return abs(thrust::get<0>(t) - thrust::get<1>(t));
  });

  // reduce: max
  return thrust::reduce(thrust::device, transform, transform + a.size(), 0.0f, thrust::maximum<float>{});
}

int main()
{
  float k = 0.5;
  float ambient_temp = 20;
  thrust::universal_vector<float> temp[] = {{42, 24, 50}, {0, 0, 0}};

  auto transformation = [=] __host__ __device__ (float temp){
    return temp + k*(ambient_temp - temp);
  };

  std::printf("step   max-change\n");

  for(int step = 0; step<3; step++)
  {
    thrust::universal_vector<float> &current = temp[step%2];
    thrust::universal_vector<float> &next = temp[(step+1)%2];
    thrust::transform(thrust::device, current.begin(), current.end(), next.begin(), transformation);

    std::printf("%d   %.2f\n", step, max_change(current, next));
  }
}

Overwriting Sources/optimized-max-diff.cpp


In [32]:
!nvcc --extended-lambda -o /tmp/a.out Sources/optimized-max-diff.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

step   max-change
0   15.00
1   7.50
2   3.75


Recall that this code is memory bound, so we'd expect that the elimination of unnecessary memory to store the difference should improve the performance. Lets evaluate this performance to see if it matches intution. Lets compare it much larger vectors.

In [34]:
%%writefile Sources/naive-vs-iterators.cpp

#include "ach.h"

float naive_max_change(const thrust::universal_vector<float>& a,
                       const thrust::universal_vector<float>& b)
{
  thrust::universal_vector<float> diff(a.size());
  thrust::transform(thrust::device, a.begin(), a.end(), b.begin(), diff.begin(),
                   [] __host__ __device__ (float x, float y){
                    return abs(x-y);
                   });

  return thrust::reduce(thrust::device, diff.begin(), diff.end(), 0.0f, thrust::maximum<float>{});
}

float max_change(const thrust::universal_vector<float>& a,
                 const thrust::universal_vector<float>& b)
{
    auto zip = thrust::make_zip_iterator(a.begin(), b.begin());
    auto transform = thrust::make_transform_iterator(zip, []__host__ __device__(thrust::tuple<float, float> t) {
        return abs(thrust::get<0>(t) - thrust::get<1>(t));
    });
    return thrust::reduce(thrust::device, transform, transform + a.size(), 0.0f, thrust::maximum<float>{});
}


int main()
{
  // allocate vectors containing 2^28 elements;
  thrust::universal_vector<float> a(1<< 28);
  thrust::universal_vector<float> b(1<< 28);

  thrust::sequence(a.begin(), a.end());
  thrust::sequence(b.begin(), b.end());

  auto start_naive = std::chrono::high_resolution_clock::now();
  naive_max_change(a, b);
  auto end_naive = std::chrono::high_resolution_clock::now();
  const double naive_duration = std::chrono::duration_cast<std::chrono::milliseconds>(end_naive - start_naive).count();

  auto start = std::chrono::high_resolution_clock::now();
  max_change(a, b);
  auto end = std::chrono::high_resolution_clock::now();
  const double duration = std::chrono::duration_cast<std::chrono::milliseconds>(end - start).count();

  std::printf("iterators are %g times faster than naive approach\n", naive_duration / duration);
}

Writing Sources/naive-vs-iterators.cpp


In [35]:
!nvcc --extended-lambda -o /tmp/a.out Sources/naive-vs-iterators.cpp -x cu -arch=native # build executable
!/tmp/a.out # run executable

iterators are 25.7778 times faster than naive approach
